# Rich Display in xeus-lfortran

xeus-lfortran supports Jupyter's rich MIME display system via the `lfortran_display` module,
injected automatically at kernel startup.

The single primitive is `display_data(mime_type, data)`.

| Call | MIME type | Use for |
|---|---|---|
| `display_data('text/html', ...)` | `text/html` | Styled text, tables, widgets |
| `display_data('image/svg+xml', ...)` | `image/svg+xml` | Vector graphics, charts |
| `display_data('text/latex', ...)` | `text/latex` | Math (rendered by MathJax) |
| `display_data('image/bmp', b64)` | `image/bmp` | Pixel images (base64-encoded) |

## 1 — HTML

In [ ]:
use lfortran_display
call display_data('text/html', &
    '<div style="font-family:sans-serif;padding:16px;border-left:4px solid #0066cc">' // &
    '<h2 style="color:#0066cc">Hello from xeus-lfortran!</h2>' // &
    '<p>This paragraph is rendered as <strong>HTML</strong> inside JupyterLab.</p>' // &
    '</div>')

## 2 — HTML table

In [ ]:
use lfortran_display
call display_data('text/html', &
    '<table style="border-collapse:collapse;font-family:sans-serif">' // &
    '<thead><tr style="background:#333;color:#fff">' // &
    '<th style="padding:8px 16px">i</th>' // &
    '<th style="padding:8px 16px">i²</th>' // &
    '<th style="padding:8px 16px">i³</th>' // &
    '</tr></thead><tbody>' // &
    '<tr><td style="padding:6px 16px">1</td><td>1</td><td>1</td></tr>' // &
    '<tr style="background:#f9f9f9"><td style="padding:6px 16px">2</td><td>4</td><td>8</td></tr>' // &
    '<tr><td style="padding:6px 16px">3</td><td>9</td><td>27</td></tr>' // &
    '<tr style="background:#f9f9f9"><td style="padding:6px 16px">4</td><td>16</td><td>64</td></tr>' // &
    '</tbody></table>')

## 3 — LaTeX (MathJax)

In [ ]:
use lfortran_display
call display_data('text/latex', '$$\int_{-\infty}^{\infty} e^{-x^2}\,dx = \sqrt{\pi}$$')

## 4 — SVG (inline vector graphic)

In [ ]:
use lfortran_display
call display_data('image/svg+xml', &
    "<svg xmlns='http://www.w3.org/2000/svg' width='540' height='100'>" // &
    "<circle cx='60'  cy='50' r='40' fill='#e74c3c' opacity='0.85'/>" // &
    "<circle cx='150' cy='50' r='40' fill='#e67e22' opacity='0.85'/>" // &
    "<circle cx='240' cy='50' r='40' fill='#f1c40f' opacity='0.85'/>" // &
    "<circle cx='330' cy='50' r='40' fill='#2ecc71' opacity='0.85'/>" // &
    "<circle cx='420' cy='50' r='40' fill='#3498db' opacity='0.85'/>" // &
    "</svg>")

## 5 — SVG bar chart

In [ ]:
use lfortran_display
call display_data('image/svg+xml', &
    "<svg xmlns='http://www.w3.org/2000/svg' width='500' height='300'" // &
    " style='background:white;font-family:Helvetica,Arial,sans-serif;'>" // &
    "<line x1='50' y1='50' x2='50' y2='250' stroke='#ccc' stroke-width='1.5'/>" // &
    "<line x1='50' y1='250' x2='470' y2='250' stroke='#ccc' stroke-width='1.5'/>" // &
    "<rect x='60'  y='91'  width='50' height='159' fill='#e74c3c' rx='4'/>" // &
    "<rect x='130' y='33'  width='50' height='217' fill='#e67e22' rx='4'/>" // &
    "<rect x='200' y='61'  width='50' height='189' fill='#f1c40f' rx='4'/>" // &
    "<rect x='270' y='10'  width='50' height='240' fill='#2ecc71' rx='4'/>" // &
    "<rect x='340' y='77'  width='50' height='173' fill='#3498db' rx='4'/>" // &
    "<rect x='410' y='118' width='50' height='132' fill='#9b59b6' rx='4'/>" // &
    "<text x='85'  y='270' text-anchor='middle' font-size='11' fill='#555'>Jan</text>" // &
    "<text x='155' y='270' text-anchor='middle' font-size='11' fill='#555'>Feb</text>" // &
    "<text x='225' y='270' text-anchor='middle' font-size='11' fill='#555'>Mar</text>" // &
    "<text x='295' y='270' text-anchor='middle' font-size='11' fill='#555'>Apr</text>" // &
    "<text x='365' y='270' text-anchor='middle' font-size='11' fill='#555'>May</text>" // &
    "<text x='435' y='270' text-anchor='middle' font-size='11' fill='#555'>Jun</text>" // &
    "</svg>")

## 6 — Pixel image: HSV colour wheel

The BMP encoder is pure Fortran — base64 encoding and BMP framing all done in user code.
The kernel only provides the generic `display_data` bridge.
White pixels outside the unit circle give the black-and-white background.

In [ ]:
! Inline BMP encoder (copy from Mandelbrot.ipynb or bmp_display.f90)
module bmp_mod
  use lfortran_display, only: display_data
  implicit none
contains
  subroutine display_image_bmp(w, h, rgba)
    integer, intent(in) :: w, h, rgba(4,w,h)
    integer :: rs, pad, fs, r, c, idx
    character(len=:), allocatable :: bmp
    rs = w*3; pad = mod(4-mod(rs,4),4); fs = 54+(rs+pad)*h
    allocate(character(len=fs) :: bmp); bmp = repeat(char(0),fs)
    bmp(1:2) = 'BM'
    call le32(bmp,3,fs); call le32(bmp,11,54)
    call le32(bmp,15,40); call le32(bmp,19,w); call le32(bmp,23,h)
    call le16(bmp,27,1); call le16(bmp,29,24)
    idx = 55
    do r = h, 1, -1
      do c = 1, w
        bmp(idx:idx)     = char(min(255,max(0,rgba(3,c,r))))
        bmp(idx+1:idx+1) = char(min(255,max(0,rgba(2,c,r))))
        bmp(idx+2:idx+2) = char(min(255,max(0,rgba(1,c,r))))
        idx = idx + 3
      end do
      idx = idx + pad
    end do
    call display_data('image/bmp', b64(bmp))
  end subroutine
  subroutine le32(s,o,v); character(len=*),intent(inout)::s; integer,intent(in)::o,v
    s(o:o)=char(iand(v,255)); s(o+1:o+1)=char(iand(ishft(v,-8),255))
    s(o+2:o+2)=char(iand(ishft(v,-16),255)); s(o+3:o+3)=char(iand(ishft(v,-24),255))
  end subroutine
  subroutine le16(s,o,v); character(len=*),intent(inout)::s; integer,intent(in)::o,v
    s(o:o)=char(iand(v,255)); s(o+1:o+1)=char(iand(ishft(v,-8),255))
  end subroutine
  function b64(data) result(out)
    character(len=*),intent(in)::data; character(len=:),allocatable::out
    character(len=64),parameter::alpha='ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/'
    integer::n,i,j,v
    n=len(data); allocate(character(len=((n+2)/3)*4)::out); j=1
    do i=1,n,3
      v=ishft(ichar(data(i:i)),16)
      if(i+1<=n) v=v+ishft(ichar(data(i+1:i+1)),8)
      if(i+2<=n) v=v+ichar(data(i+2:i+2))
      out(j:j)=alpha(ishft(v,-18)+1:ishft(v,-18)+1)
      out(j+1:j+1)=alpha(iand(ishft(v,-12),63)+1:iand(ishft(v,-12),63)+1)
      out(j+2:j+2)=merge(alpha(iand(ishft(v,-6),63)+1:iand(ishft(v,-6),63)+1),'=',i+1<=n)
      out(j+3:j+3)=merge(alpha(iand(v,63)+1:iand(v,63)+1),'=',i+2<=n)
      j=j+4
    end do
  end function
end module bmp_mod

In [ ]:
use bmp_mod
integer, parameter :: W = 256, H = 256
integer :: pixels(4,W,H), ci, cj, hi
real :: cx, cy, dist, hue, r, g, b, sat, val, f, p, q, t

do cj = 1, H
  do ci = 1, W
    cx = (ci - W/2.0) / (W/2.0)
    cy = (cj - H/2.0) / (H/2.0)
    dist = sqrt(cx**2 + cy**2)
    if (dist > 1.0) then
      pixels(:,ci,cj) = [255, 255, 255, 255]  ! white outside circle
    else
      hue = atan2(cy, cx) / (2.0 * 3.14159265) + 0.5
      sat = dist; val = 1.0
      hi = int(hue * 6)
      f = hue*6 - hi; p = val*(1-sat); q = val*(1-f*sat); t = val*(1-(1-f)*sat)
      if     (mod(hi,6)==0) then; r=val; g=t;   b=p
      else if(mod(hi,6)==1) then; r=q;   g=val; b=p
      else if(mod(hi,6)==2) then; r=p;   g=val; b=t
      else if(mod(hi,6)==3) then; r=p;   g=q;   b=val
      else if(mod(hi,6)==4) then; r=t;   g=p;   b=val
      else;                       r=val; g=p;   b=q
      end if
      pixels(1,ci,cj)=int(r*255); pixels(2,ci,cj)=int(g*255)
      pixels(3,ci,cj)=int(b*255); pixels(4,ci,cj)=255
    end if
  end do
end do
call display_image_bmp(W, H, pixels)